In [ ]:
%run ./1_config


In [ ]:

import time
from pyspark.sql import functions as F

class SetupHelper:
    def __init__(self):
        self.c = conf

    def create_tables(self):
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS {self.c.lakehouse}.{self.c.schema}")
        spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {self.c.table_fqn(self.c.bronze_events)} (
          event_id STRING, session_id STRING, event_ts TIMESTAMP, event_type STRING,
          user_id STRING, ip STRING, user_agent STRING, amount DOUBLE,
          event_date DATE, _ingested_at TIMESTAMP
        ) USING DELTA PARTITIONED BY (event_date)
        """)
        spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {self.c.table_fqn(self.c.bronze_ipstack_raw)} (
          ip STRING, response_json STRING, http_status INT, api_called_at TIMESTAMP
        ) USING DELTA
        """)
        spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {self.c.table_fqn(self.c.bronze_ipstack_errors)} (
          ip STRING, error_message STRING, http_status INT, api_called_at TIMESTAMP
        ) USING DELTA
        """)
        spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {self.c.table_fqn(self.c.silver_ip_dim)} (
          ip STRING, country_code STRING, country_name STRING, region STRING, city STRING,
          latitude DOUBLE, longitude DOUBLE, timezone_id STRING, currency_code STRING,
          isp STRING, org STRING, is_proxy BOOLEAN, is_vpn BOOLEAN, threat_level STRING,
          enriched_at TIMESTAMP
        ) USING DELTA
        """)
        spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {self.c.table_fqn(self.c.silver_events_enriched)} (
          event_id STRING, session_id STRING, event_ts TIMESTAMP, event_type STRING,
          user_id STRING, ip STRING, user_agent STRING, amount DOUBLE, event_date DATE,
          country_code STRING, country_name STRING, timezone_id STRING, isp STRING,
          is_vpn BOOLEAN, device STRING, spend DOUBLE
        ) USING DELTA PARTITIONED BY (event_date)
        """)
        spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {self.c.table_fqn(self.c.gold_geo_traffic_daily)} (
          date DATE, country_code STRING, event_count BIGINT, unique_sessions BIGINT, revenue DOUBLE
        ) USING DELTA
        """)
        spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {self.c.table_fqn(self.c.gold_fraud_signals)} (
          session_id STRING, ip STRING, country_code STRING, is_vpn BOOLEAN,
          vpn_risk_score INT, event_count_1h BIGINT, flag_suspicious BOOLEAN
        ) USING DELTA
        """)
        spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {self.c.table_fqn(self.c.gold_customer_features)} (
          user_id STRING, primary_country STRING, timezone STRING, total_spend DOUBLE,
          session_count BIGINT, last_seen_at TIMESTAMP
        ) USING DELTA
        """)

    def setup(self):
        t0 = time.time()
        self.create_tables()
        print(f"✅ Setup completed in {int(time.time()-t0)}s")

SetupHelper().setup()

